## EVALUACIÓN EN VALDO COMPLETO (72 imágenes)

### D201 (trained only on ADNI sCMBs)

In [11]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_201_3dfullres_smallpatch"
GT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds/nnUNet_raw_data/Dataset881_eval_VALDO/labelsTs"
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/VALDO_analysis_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n===== DEBUG INFO =====")
print(f"PRED_DIR existe: {os.path.exists(PRED_DIR)}")
print(f"GT_DIR existe: {os.path.exists(GT_DIR)}")
print(f"OUTPUT_DIR existe: {os.path.exists(OUTPUT_DIR)}")

pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])

print(f"\nNúmero de predicciones: {len(pred_files)}")
print(f"Número de GT: {len(gt_files)}")

# Ver intersección real de nombres
common_files = sorted(list(set(pred_files) & set(gt_files)))
print(f"Archivos en común: {len(common_files)}")

if len(common_files) == 0:
    print("ERROR: No hay nombres coincidentes entre predicciones y GT")
    print("Ejemplo PRED:", pred_files[:5])
    print("Ejemplo GT:", gt_files[:5])
    exit()

results_list = []

# ========================= PROCESAMIENTO =========================
for fname in common_files:

    print(f"\n--- Procesando: {fname} ---")

    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    if not os.path.exists(pred_path):
        print(f"Pred no encontrado: {pred_path}")
        continue

    if not os.path.exists(gt_path):
        print(f"GT no encontrado: {gt_path}")
        continue

    try:
        pred_img = nib.load(pred_path)
        gt_img = nib.load(gt_path)
    except Exception as e:
        print(f"Error cargando NIfTI: {e}")
        continue

    pred = pred_img.get_fdata()
    gt = gt_img.get_fdata()

    print(f"Shape pred: {pred.shape} | Shape GT: {gt.shape}")

    if pred.shape != gt.shape:
        print("⚠️ ERROR: Shapes diferentes → se ignora")
        continue

    # Binarizar
    pred_bin = pred > 0
    gt_bin = gt > 0

    # Chequeo contenido
    if np.sum(pred_bin) == 0:
        print("Predicción vacía (todo 0)")

    if np.sum(gt_bin) == 0:
        print("GT vacío (todo 0)")

    # Etiquetado
    labeled_pred, num_pred = label(pred_bin, return_num=True)
    labeled_gt, num_gt = label(gt_bin, return_num=True)

    print(f"Blobs GT: {num_gt} | Blobs Pred: {num_pred}")

    # ========================= MATCHING =========================
    tp_count = 0
    used_pred_blobs = set()

    for gt_id in range(1, num_gt + 1):

        gt_blob = (labeled_gt == gt_id)

        overlapping_pred_ids = np.unique(labeled_pred[gt_blob])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        if len(overlapping_pred_ids) > 0:
            print(f"  GT {gt_id} toca pred blobs: {overlapping_pred_ids}")

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break

        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    # ========================= MÉTRICAS =========================
    fp = num_pred - len(used_pred_blobs)
    fn = num_gt - tp_count

    precision = tp_count / num_pred if num_pred > 0 else 0
    recall = tp_count / num_gt if num_gt > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"TP: {tp_count} | FP: {fp} | FN: {fn}")

    results_list.append({
        'ID': fname,
        'GT_blobs': num_gt,
        'Pred_blobs_total': num_pred,
        'TP': tp_count,
        'FP': fp,
        'FN': fn,
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1-Score': round(f1, 4)
    })

# ========================= SALIDA =========================
df_res = pd.DataFrame(results_list)

csv_path = os.path.join(OUTPUT_DIR, "VALDO_blob_analysis_DEBUG.csv")
df_res.to_csv(csv_path, index=False)

print("\n===== RESUMEN FINAL =====")
if len(df_res) > 0:
    print(f"Media Recall: {df_res['Recall'].mean():.4f}")
    print(f"Media Precision: {df_res['Precision'].mean():.4f}")
    print(f"Media F1: {df_res['F1-Score'].mean():.4f}")
else:
    print("No se procesó ningún caso")

print(f"CSV guardado en: {csv_path}")


===== DEBUG INFO =====
PRED_DIR existe: True
GT_DIR existe: True
OUTPUT_DIR existe: True

Número de predicciones: 72
Número de GT: 72
Archivos en común: 72

--- Procesando: sub-101.nii.gz ---
Shape pred: (512, 512, 35) | Shape GT: (512, 512, 35)
Predicción vacía (todo 0)
Blobs GT: 12 | Blobs Pred: 0
TP: 0 | FP: 0 | FN: 12

--- Procesando: sub-102.nii.gz ---
Shape pred: (512, 512, 35) | Shape GT: (512, 512, 35)
Predicción vacía (todo 0)
Blobs GT: 1 | Blobs Pred: 0
TP: 0 | FP: 0 | FN: 1

--- Procesando: sub-103.nii.gz ---
Shape pred: (512, 512, 35) | Shape GT: (512, 512, 35)
Blobs GT: 6 | Blobs Pred: 1
  GT 2 toca pred blobs: [1]
TP: 1 | FP: 0 | FN: 5

--- Procesando: sub-104.nii.gz ---
Shape pred: (512, 512, 35) | Shape GT: (512, 512, 35)
Predicción vacía (todo 0)
GT vacío (todo 0)
Blobs GT: 0 | Blobs Pred: 0
TP: 0 | FP: 0 | FN: 0

--- Procesando: sub-105.nii.gz ---
Shape pred: (512, 512, 35) | Shape GT: (512, 512, 35)
GT vacío (todo 0)
Blobs GT: 0 | Blobs Pred: 1
TP: 0 | FP: 1 | FN: 0

In [12]:
# ========================= TOTALES GLOBALES =========================
if len(results_list) > 0:
    total_tp = sum(r['TP'] for r in results_list)
    total_fp = sum(r['FP'] for r in results_list)
    total_fn = sum(r['FN'] for r in results_list)

    total_pred = sum(r['Pred_blobs_total'] for r in results_list)
    total_gt = sum(r['GT_blobs'] for r in results_list)

    # Métricas globales (micro-average)
    precision_global = total_tp / total_pred if total_pred > 0 else 0
    recall_global = total_tp / total_gt if total_gt > 0 else 0
    f1_global = (2 * precision_global * recall_global / (precision_global + recall_global)
                 if (precision_global + recall_global) > 0 else 0)

    print("\n===== MÉTRICAS GLOBALES D201 en VALDO =====")
    print(f"Total GT (lesiones reales): {total_gt}")
    print(f"Total Pred (lesiones detectadas): {total_pred}")
    print(f"TP totales: {total_tp}")
    print(f"FP totales: {total_fp}")
    print(f"FN totales: {total_fn}")
    print("-" * 40)
    print(f"Precision global: {precision_global:.4f}")
    print(f"Recall global (Sensitivity): {recall_global:.4f}")
    print(f"F1-score global: {f1_global:.4f}")
else:
    print("⚠️ No hay resultados para calcular métricas globales")


===== MÉTRICAS GLOBALES D201 en VALDO =====
Total GT (lesiones reales): 236
Total Pred (lesiones detectadas): 30
TP totales: 11
FP totales: 19
FN totales: 225
----------------------------------------
Precision global: 0.3667
Recall global (Sensitivity): 0.0466
F1-score global: 0.0827


In [13]:
import pandas as pd

# ========================= ASIGNAR COHORTE =========================
def get_cohort(fname):
    if fname.startswith("sub-1"):
        return "SABRE"
    elif fname.startswith("sub-2"):
        return "RSS"
    elif fname.startswith("sub-3"):
        return "ALFA"
    else:
        return "UNKNOWN"

df_res['Cohort'] = df_res['ID'].apply(get_cohort)

# ========================= CÁLCULO POR COHORTE =========================
cohort_results = []

for cohort, df_c in df_res.groupby('Cohort'):

    # --- Totales (micro) ---
    total_tp = df_c['TP'].sum()
    total_fp = df_c['FP'].sum()
    total_fn = df_c['FN'].sum()

    total_pred = df_c['Pred_blobs_total'].sum()
    total_gt = df_c['GT_blobs'].sum()

    precision_micro = total_tp / total_pred if total_pred > 0 else 0
    recall_micro = total_tp / total_gt if total_gt > 0 else 0
    f1_micro = (2 * precision_micro * recall_micro / (precision_micro + recall_micro)
                if (precision_micro + recall_micro) > 0 else 0)

    # --- Medias (macro) ---
    precision_macro = df_c['Precision'].mean()
    recall_macro = df_c['Recall'].mean()
    f1_macro = df_c['F1-Score'].mean()

    cohort_results.append({
        'Cohort': cohort,
        'N_subjects': len(df_c),

        # Totales
        'Total_GT': total_gt,
        'Total_Pred': total_pred,
        'TP': total_tp,
        'FP': total_fp,
        'FN': total_fn,

        # Micro
        'Precision_micro': round(precision_micro, 4),
        'Recall_micro': round(recall_micro, 4),
        'F1_micro': round(f1_micro, 4),

        # Macro
        'Precision_macro': round(precision_macro, 4),
        'Recall_macro': round(recall_macro, 4),
        'F1_macro': round(f1_macro, 4),
    })

# ========================= RESULTADO =========================
df_cohort = pd.DataFrame(cohort_results)

print("\n===== RESULTADOS POR COHORTE =====")
print(df_cohort)

# Guardar CSV
output_path = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/VALDO_analysis_results/cohort_metrics.csv"
df_cohort.to_csv(output_path, index=False)

print(f"\nCSV guardado en: {output_path}")


===== RESULTADOS POR COHORTE =====
  Cohort  N_subjects  Total_GT  Total_Pred  TP  FP   FN  Precision_micro  \
0   ALFA          27        34           7   3   4   31           0.4286   
1    RSS          34        96          19   6  13   90           0.3158   
2  SABRE          11       106           4   2   2  104           0.5000   

   Recall_micro  F1_micro  Precision_macro  Recall_macro  F1_macro  
0        0.0882    0.1463           0.1111        0.0926    0.0988  
1        0.0625    0.1043           0.0735        0.0125    0.0213  
2        0.0189    0.0364           0.1818        0.0455    0.0714  

CSV guardado en: /media/PORT-DISK/Practicas/MicroBleeds_Generation/VALDO_analysis_results/cohort_metrics.csv


### D202 en VALDO

In [8]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_202_3dfullres_smallpatch"
GT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds/nnUNet_raw_data/Dataset881_eval_VALDO/labelsTs"
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/VALDO_analysis_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n===== DEBUG INFO =====")
print(f"PRED_DIR existe: {os.path.exists(PRED_DIR)}")
print(f"GT_DIR existe: {os.path.exists(GT_DIR)}")
print(f"OUTPUT_DIR existe: {os.path.exists(OUTPUT_DIR)}")

pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])

print(f"\nNúmero de predicciones: {len(pred_files)}")
print(f"Número de GT: {len(gt_files)}")

# Ver intersección real de nombres
common_files = sorted(list(set(pred_files) & set(gt_files)))
print(f"Archivos en común: {len(common_files)}")

if len(common_files) == 0:
    print("ERROR: No hay nombres coincidentes entre predicciones y GT")
    print("Ejemplo PRED:", pred_files[:5])
    print("Ejemplo GT:", gt_files[:5])
    exit()

results_list = []

# ========================= PROCESAMIENTO =========================
for fname in common_files:

    print(f"\n--- Procesando: {fname} ---")

    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    if not os.path.exists(pred_path):
        print(f"Pred no encontrado: {pred_path}")
        continue

    if not os.path.exists(gt_path):
        print(f"GT no encontrado: {gt_path}")
        continue

    try:
        pred_img = nib.load(pred_path)
        gt_img = nib.load(gt_path)
    except Exception as e:
        print(f"Error cargando NIfTI: {e}")
        continue

    pred = pred_img.get_fdata()
    gt = gt_img.get_fdata()

    print(f"Shape pred: {pred.shape} | Shape GT: {gt.shape}")

    if pred.shape != gt.shape:
        print("⚠️ ERROR: Shapes diferentes → se ignora")
        continue

    # Binarizar
    pred_bin = pred > 0
    gt_bin = gt > 0

    # Chequeo contenido
    if np.sum(pred_bin) == 0:
        print("Predicción vacía (todo 0)")

    if np.sum(gt_bin) == 0:
        print("GT vacío (todo 0)")

    # Etiquetado
    labeled_pred, num_pred = label(pred_bin, return_num=True)
    labeled_gt, num_gt = label(gt_bin, return_num=True)

    print(f"Blobs GT: {num_gt} | Blobs Pred: {num_pred}")

    # ========================= MATCHING =========================
    tp_count = 0
    used_pred_blobs = set()

    for gt_id in range(1, num_gt + 1):

        gt_blob = (labeled_gt == gt_id)

        overlapping_pred_ids = np.unique(labeled_pred[gt_blob])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        if len(overlapping_pred_ids) > 0:
            print(f"  GT {gt_id} toca pred blobs: {overlapping_pred_ids}")

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break

        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    # ========================= MÉTRICAS =========================
    fp = num_pred - len(used_pred_blobs)
    fn = num_gt - tp_count

    precision = tp_count / num_pred if num_pred > 0 else 0
    recall = tp_count / num_gt if num_gt > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"TP: {tp_count} | FP: {fp} | FN: {fn}")

    results_list.append({
        'ID': fname,
        'GT_blobs': num_gt,
        'Pred_blobs_total': num_pred,
        'TP': tp_count,
        'FP': fp,
        'FN': fn,
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1-Score': round(f1, 4)
    })

# ========================= SALIDA =========================
df_res = pd.DataFrame(results_list)

csv_path = os.path.join(OUTPUT_DIR, "VALDO_blob_analysis_DEBUG_202.csv")
df_res.to_csv(csv_path, index=False)

print("\n===== RESUMEN FINAL =====")
if len(df_res) > 0:
    print(f"Media Recall: {df_res['Recall'].mean():.4f}")
    print(f"Media Precision: {df_res['Precision'].mean():.4f}")
    print(f"Media F1: {df_res['F1-Score'].mean():.4f}")
else:
    print("No se procesó ningún caso")

print(f"CSV guardado en: {csv_path}")


===== DEBUG INFO =====
PRED_DIR existe: True
GT_DIR existe: True
OUTPUT_DIR existe: True

Número de predicciones: 72
Número de GT: 72
Archivos en común: 72

--- Procesando: sub-101.nii.gz ---
Shape pred: (512, 512, 35) | Shape GT: (512, 512, 35)
Blobs GT: 12 | Blobs Pred: 3
  GT 8 toca pred blobs: [3]
TP: 1 | FP: 2 | FN: 11

--- Procesando: sub-102.nii.gz ---
Shape pred: (512, 512, 35) | Shape GT: (512, 512, 35)
Blobs GT: 1 | Blobs Pred: 2
  GT 1 toca pred blobs: [2]
TP: 1 | FP: 1 | FN: 0

--- Procesando: sub-103.nii.gz ---
Shape pred: (512, 512, 35) | Shape GT: (512, 512, 35)
Blobs GT: 6 | Blobs Pred: 2
  GT 4 toca pred blobs: [1]
  GT 6 toca pred blobs: [2]
TP: 2 | FP: 0 | FN: 4

--- Procesando: sub-104.nii.gz ---
Shape pred: (512, 512, 35) | Shape GT: (512, 512, 35)
Predicción vacía (todo 0)
GT vacío (todo 0)
Blobs GT: 0 | Blobs Pred: 0
TP: 0 | FP: 0 | FN: 0

--- Procesando: sub-105.nii.gz ---
Shape pred: (512, 512, 35) | Shape GT: (512, 512, 35)
GT vacío (todo 0)
Blobs GT: 0 | Blo

In [9]:
# ========================= TOTALES GLOBALES =========================
if len(results_list) > 0:
    total_tp = sum(r['TP'] for r in results_list)
    total_fp = sum(r['FP'] for r in results_list)
    total_fn = sum(r['FN'] for r in results_list)

    total_pred = sum(r['Pred_blobs_total'] for r in results_list)
    total_gt = sum(r['GT_blobs'] for r in results_list)

    # Métricas globales (micro-average)
    precision_global = total_tp / total_pred if total_pred > 0 else 0
    recall_global = total_tp / total_gt if total_gt > 0 else 0
    f1_global = (2 * precision_global * recall_global / (precision_global + recall_global)
                 if (precision_global + recall_global) > 0 else 0)

    print("\n===== MÉTRICAS GLOBALES D202 en VALDO =====")
    print(f"Total GT (lesiones reales): {total_gt}")
    print(f"Total Pred (lesiones detectadas): {total_pred}")
    print(f"TP totales: {total_tp}")
    print(f"FP totales: {total_fp}")
    print(f"FN totales: {total_fn}")
    print("-" * 40)
    print(f"Precision global: {precision_global:.4f}")
    print(f"Recall global (Sensitivity): {recall_global:.4f}")
    print(f"F1-score global: {f1_global:.4f}")
else:
    print("No hay resultados para calcular métricas globales")


===== MÉTRICAS GLOBALES D202 en VALDO =====
Total GT (lesiones reales): 236
Total Pred (lesiones detectadas): 219
TP totales: 89
FP totales: 130
FN totales: 147
----------------------------------------
Precision global: 0.4064
Recall global (Sensitivity): 0.3771
F1-score global: 0.3912


In [10]:
import pandas as pd

# ========================= ASIGNAR COHORTE =========================
def get_cohort(fname):
    if fname.startswith("sub-1"):
        return "SABRE"
    elif fname.startswith("sub-2"):
        return "RSS"
    elif fname.startswith("sub-3"):
        return "ALFA"
    else:
        return "UNKNOWN"

df_res['Cohort'] = df_res['ID'].apply(get_cohort)

# ========================= CÁLCULO POR COHORTE =========================
cohort_results = []

for cohort, df_c in df_res.groupby('Cohort'):

    # --- Totales (micro) ---
    total_tp = df_c['TP'].sum()
    total_fp = df_c['FP'].sum()
    total_fn = df_c['FN'].sum()

    total_pred = df_c['Pred_blobs_total'].sum()
    total_gt = df_c['GT_blobs'].sum()

    precision_micro = total_tp / total_pred if total_pred > 0 else 0
    recall_micro = total_tp / total_gt if total_gt > 0 else 0
    f1_micro = (2 * precision_micro * recall_micro / (precision_micro + recall_micro)
                if (precision_micro + recall_micro) > 0 else 0)

    # --- Medias (macro) ---
    precision_macro = df_c['Precision'].mean()
    recall_macro = df_c['Recall'].mean()
    f1_macro = df_c['F1-Score'].mean()

    cohort_results.append({
        'Cohort': cohort,
        'N_subjects': len(df_c),

        # Totales
        'Total_GT': total_gt,
        'Total_Pred': total_pred,
        'TP': total_tp,
        'FP': total_fp,
        'FN': total_fn,

        # Micro
        'Precision_micro': round(precision_micro, 4),
        'Recall_micro': round(recall_micro, 4),
        'F1_micro': round(f1_micro, 4),

        # Macro
        'Precision_macro': round(precision_macro, 4),
        'Recall_macro': round(recall_macro, 4),
        'F1_macro': round(f1_macro, 4),
    })

# ========================= RESULTADO =========================
df_cohort = pd.DataFrame(cohort_results)

print("\n===== RESULTADOS POR COHORTE =====")
print(df_cohort)

# Guardar CSV
output_path = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/VALDO_analysis_results/cohort_metrics.csv"
df_cohort.to_csv(output_path, index=False)

print(f"\nCSV guardado en: {output_path}")


===== RESULTADOS POR COHORTE =====
  Cohort  N_subjects  Total_GT  Total_Pred  TP  FP  FN  Precision_micro  \
0   ALFA          27        34          47  22  25  12           0.4681   
1    RSS          34        96         130  35  95  61           0.2692   
2  SABRE          11       106          42  32  10  74           0.7619   

   Recall_micro  F1_micro  Precision_macro  Recall_macro  F1_macro  
0        0.6471    0.5432           0.5006        0.6481    0.5340  
1        0.3646    0.3097           0.1511        0.1240    0.1210  
2        0.3019    0.4324           0.4739        0.2925    0.3226  

CSV guardado en: /media/PORT-DISK/Practicas/MicroBleeds_Generation/VALDO_analysis_results/cohort_metrics.csv
